# 02 TranFormData

Notebook นี้ใช้สำหรับแปลงข้อมูลจากไฟล์ raw ก่อนเข้าสู่ขั้น clean data

In [99]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [100]:
import numpy as np
import pandas as pd

from src.utils.paths import RAW_DATA_DIR, INTERIM_DATA_DIR

RAW_DATA_DIR, INTERIM_DATA_DIR

(WindowsPath('D:/ML/data/raw'), WindowsPath('D:/ML/data/interim'))

## Load Raw Data

ใช้ไฟล์ raw `vwTimeStampDashbaord.csv` เป็นจุดเริ่มต้น แล้วค่อยแปลงค่าใน notebook นี้

In [101]:
source_path = RAW_DATA_DIR / "vwTimeStampDashbaord.csv"
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f"source: {source_path}")
print(f"shape: {df.shape}")
df.head()

source: D:\ML\data\raw\vwTimeStampDashbaord.csv
shape: (10000, 40)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,Walk-in,10/04/2026 10:01:58,9,1000000005,71-2054,SB1PL260410025,บ.ส.เอื้อสุข จก.,10/04/2026 10:02:03,N,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SB1,Walk-in,10/04/2026 09:58:49,8,1000000008,700-8513,SB1PL260410024,มบูเครือวัลย์ Lake and Park ลำผักชี,10/04/2026 09:59:06,N,...,34,NaN,NaN,NaN,NaN,10/04/2026 10:13:25,10/04/2026 10:20:44,10/04/2026 10:11:37,NaN,NaN
2,SB1,Walk-in,10/04/2026 09:57:56,7,1000000003,89-4045,SB1PL260410023,นครปฐม บางเลน,10/04/2026 09:57:58,N,...,0,NaN,NaN,NaN,NaN,10/04/2026 09:58:32,10/04/2026 09:59:48,NaN,NaN,NaN
3,SB1,Walk-in,10/04/2026 09:56:17,6,1000000005,80-8223,SB1PL260410022,บ.อุดมชัยกู๊ดโฮมสมุทรสงคราม จก.,10/04/2026 09:56:23,N,...,0,NaN,NaN,10/04/2026 10:19:38,10/04/2026 10:20:33,10/04/2026 10:00:25,10/04/2026 10:08:47,NaN,NaN,NaN
4,SB1,Walk-in,10/04/2026 09:48:27,5,1000000003,89-4045,SB1PL260410021,บ.จำหน่ายวัตถุก่อสร้าง จก.,10/04/2026 09:48:32,N,...,70,NaN,NaN,NaN,NaN,10/04/2026 09:49:36,NaN,10/04/2026 10:05:27,10/04/2026 10:10:23,NaN


## Quick Check in Raw File

ดูค่าที่น่าสงสัยในไฟล์ raw ก่อนแปลงข้อมูล

In [102]:
for col in ['PickListType', 'PrepareForward', 'CarType', 'TruckStatus', 'PackListStatus']:
    print(f"\n{col}")
    print(df[col].astype(str).value_counts(dropna=False).head(10))


PickListType
PickListType
Walk-in    9538
SmartQ      462
Name: count, dtype: int64

PrepareForward
PrepareForward
N              5775
Y              4219
46106.72737       1
46106.71925       1
46106.71063       1
46106.71279       1
46098.60498       1
45975.57596       1
Name: count, dtype: int64

CarType
CarType
1000000003    5188
1000000005    3305
1000000008     990
1000000004     296
1000000006      64
1000000009      64
1000000001      58
1000000000      15
1000000002      11
6                6
Name: count, dtype: int64

TruckStatus
TruckStatus
Loading        9723
Waiting         271
46111.69963       1
46111.66447       1
46111.54631       1
46111.50123       1
46098.61959       1
NaN               1
Name: count, dtype: int64

PackListStatus
PackListStatus
OPERATORCOMPLETED    9654
REMOVE                303
WAITCHECKER            32
Loading                 6
CHECKERASSIGN           5
Name: count, dtype: int64


In [103]:
df.loc[
    df['PackListNo'].isna(),
    ['PickDate', 'CarNo', 'PackListNo', 'CustomerName', 'QueueTime', 'PrepareForward', 'TruckStatus', 'PackListStatus']
].head(10)

,PickDate,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,TruckStatus,PackListStatus
993,30/03/2026 16:22:06,700-8556 ตู้8/ฟิลิปปินส์,NaN,SB1PL260325076,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.72737,46111.69963,Loading
1008,30/03/2026 15:18:18,701-0738 ตู้6/ฟิลิปปินส์,NaN,SB1PL260325073,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.71925,46111.66447,Loading
1039,30/03/2026 12:37:52,72-2248 ตู้3/ฟิลิปปินส์,NaN,SB1PL260325070,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.71063,46111.54631,Loading
1047,30/03/2026 11:17:33,700-0959 ตู้4/ฟิลิปปินส์,NaN,SB1PL260325071,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.71279,46111.50123,Loading
2111,17/03/2026 14:31:02,70-1312 ชลบุรี,NaN,SB1PL260317049,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",46098.60498,46098.61959,Loading
9646,14/11/2025 13:49:18,72-0128ทุ่งสง,NaN,SB1PL251114049,SCGR Nakhon Srithammarat Plant SCG Roofing Co....,45975.57596,NaN,Loading


## Mapping Rules

แปลงค่าของ `CarType` และ `PickListType` กลับลงคอลัมน์เดิม

In [104]:
CAR_TYPE_MAP = {
    4: '4 ล้อ', 1000000008: '4 ล้อ',
    6: '6 ล้อ', 1000000003: '6 ล้อ',
    10: '10 ล้อ', 1000000000: '10 ล้อ', 1000000004: '10 ล้อ',
    18: 'เทรเลอร์', 22: 'เทรเลอร์', 1000000001: 'เทรเลอร์',
    1000000002: 'เทรเลอร์', 1000000005: 'เทรเลอร์', 1000000006: 'เทรเลอร์',
    1000000007: 'เทรเลอร์', 1000000009: 'เทรเลอร์',
    1000000010: 'อื่นๆ',
}


def map_pick_list_type(row):
    if row.get('PickListType') == 'SmartQ':
        return 'SmartQ'
    if row.get('PickListType') == 'Walk-in':
        return 'ล่วงหน้า' if row.get('PrepareForward') == 'Y' else 'Walk in'
    return 'Unknown'


def map_car_type(value):
    try:
        return CAR_TYPE_MAP.get(int(value), 'Unknown') if pd.notna(value) else 'Unknown'
    except (TypeError, ValueError):
        return 'Unknown'

## Apply Category Transform

แทนค่าใน `CarType` และ `PickListType` แล้วคงข้อมูลอื่นไว้เหมือนเดิม

In [105]:
df_transform = df.copy()

df_transform['CarType'] = df_transform['CarType'].apply(map_car_type)
df_transform['PickListType'] = df_transform.apply(map_pick_list_type, axis=1)

df_transform[['CarType', 'PickListType', 'PrepareForward']].head(10)

,CarType,PickListType,PrepareForward
0,เทรเลอร์,Walk in,N
1,4 ล้อ,Walk in,N
2,6 ล้อ,Walk in,N
3,เทรเลอร์,Walk in,N
4,6 ล้อ,Walk in,N
5,เทรเลอร์,ล่วงหน้า,Y
6,เทรเลอร์,ล่วงหน้า,Y
7,เทรเลอร์,ล่วงหน้า,Y
8,เทรเลอร์,Walk in,N
9,6 ล้อ,Walk in,N


## Normalize Missing-like Values

แปลงค่าอย่าง `NULL`, ช่องว่าง, และ `nan` ให้เป็น `NaN` มาตรฐาน

In [106]:
NULL_LIKE_VALUES = ['NULL', '', 'nan', 'NaN', 'None']
df_transform = df_transform.replace(NULL_LIKE_VALUES, np.nan)

df_transform.isna().sum().sort_values(ascending=False).head(15)

Unnamed: 39            10000
TruckOverTimeRemark     9964
TruckOverTimeName       9955
AccEnd                  5175
AccStart                5154
FittingEnd              4252
FittingStart            4241
TileEnd                 1831
TileStart               1822
OperatorCarConfirm       658
FirstPostPallet          468
LastPostPallet           468
PostingTime              350
CarConfirm               271
PostLocationName         261
dtype: int64

## Convert Datetime and Numeric Columns

แปลงคอลัมน์เวลาเป็น `datetime` และคอลัมน์ตัวเลขเป็น `numeric`

In [107]:
datetime_columns = [
    'PickDate', 'QueueTime', 'CreateDate', 'PickingTime', 'OperatorCarConfirm',
    'CarConfirm', 'PostingTime', 'FirstPostPallet', 'LastPostPallet',
    'TileStart', 'TileEnd', 'FittingStart', 'FittingEnd', 'AccStart', 'AccEnd', 'TruckReceiveDate'
]

numeric_columns = [
    'TruckSeqNo', 'TruckReceiveHour', 'TruckReceiveMinute',
    'CPACTileSapAmount', 'PRESTIGETileSapAmount', 'NEUSTILETileSapAmount',
    'CPACFittingSapAmount', 'PRESTIGEFittingSapAmount', 'NEUSTILEFittingSapAmount',
    'DURAFittingSapAmount', 'ACCESSORIESSapAmount'
]

int64_columns = ['CPACTileSapAmount']

for col in datetime_columns:
    df_transform[col] = pd.to_datetime(
        df_transform[col],
        format='%d/%m/%Y %H:%M:%S',
        errors='coerce'
    )

for col in numeric_columns:
    df_transform[col] = pd.to_numeric(df_transform[col], errors='coerce')

for col in int64_columns:
    df_transform[col] = df_transform[col].astype('Int64')

df_transform.dtypes

PlantName                              str
PickListType                           str
PickDate                    datetime64[us]
TruckSeqNo                           int64
CarType                                str
CarNo                                  str
PackListNo                             str
CustomerName                           str
QueueTime                   datetime64[us]
PrepareForward                         str
TruckReceiveDate            datetime64[us]
TruckReceiveHour                     int64
TruckReceiveMinute                   int64
CreateDate                  datetime64[us]
PickingTime                 datetime64[us]
OperatorCarConfirm          datetime64[us]
CarConfirm                  datetime64[us]
PostingTime                 datetime64[us]
FirstPostPallet             datetime64[us]
LastPostPallet              datetime64[us]
TruckStatus                            str
PackListStatus                         str
PostLocationName                       str
CPACTileSap

## Check Converted Data

สรุป dtype หลังแปลง และเช็กจำนวนข้อมูลที่ parse ได้

In [108]:
dtype_summary = pd.DataFrame(
    {
        'column': df_transform.columns,
        'dtype': df_transform.dtypes.astype(str).values,
    }
)

dtype_summary.sort_values(['dtype', 'column']).reset_index(drop=True)

,column,dtype
0,CPACTileSapAmount,Int64
1,AccEnd,datetime64[us]
2,AccStart,datetime64[us]
3,CarConfirm,datetime64[us]
4,CreateDate,datetime64[us]
5,FirstPostPallet,datetime64[us]
6,FittingEnd,datetime64[us]
7,FittingStart,datetime64[us]
8,LastPostPallet,datetime64[us]
9,OperatorCarConfirm,datetime64[us]


In [109]:
datetime_check = pd.DataFrame(
    {
        'column': datetime_columns,
        'non_null_after_parse': [df_transform[col].notna().sum() for col in datetime_columns],
        'missing_after_parse': [df_transform[col].isna().sum() for col in datetime_columns],
    }
)

datetime_check

,column,non_null_after_parse,missing_after_parse
0,PickDate,10000,0
1,QueueTime,9952,48
2,CreateDate,10000,0
3,PickingTime,10000,0
4,OperatorCarConfirm,9342,658
5,CarConfirm,9729,271
6,PostingTime,9650,350
7,FirstPostPallet,9532,468
8,LastPostPallet,9532,468
9,TileStart,8178,1822


In [110]:
numeric_check = pd.DataFrame(
    {
        'column': numeric_columns,
        'non_null_after_parse': [df_transform[col].notna().sum() for col in numeric_columns],
        'missing_after_parse': [df_transform[col].isna().sum() for col in numeric_columns],
    }
)

numeric_check

,column,non_null_after_parse,missing_after_parse
0,TruckSeqNo,10000,0
1,TruckReceiveHour,10000,0
2,TruckReceiveMinute,10000,0
3,CPACTileSapAmount,9994,6
4,PRESTIGETileSapAmount,10000,0
5,NEUSTILETileSapAmount,10000,0
6,CPACFittingSapAmount,10000,0
7,PRESTIGEFittingSapAmount,10000,0
8,NEUSTILEFittingSapAmount,10000,0
9,DURAFittingSapAmount,10000,0


In [111]:
df_transform.head()

,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,Walk in,2026-04-10 10:01:58,9,เทรเลอร์,71-2054,SB1PL260410025,บ.ส.เอื้อสุข จก.,2026-04-10 10:02:03,N,...,0,NaN,NaN,NaT,NaT,NaT,NaT,NaT,NaT,NaN
1,SB1,Walk in,2026-04-10 09:58:49,8,4 ล้อ,700-8513,SB1PL260410024,มบูเครือวัลย์ Lake and Park ลำผักชี,2026-04-10 09:59:06,N,...,34,NaN,NaN,NaT,NaT,2026-04-10 10:13:25,2026-04-10 10:20:44,2026-04-10 10:11:37,NaT,NaN
2,SB1,Walk in,2026-04-10 09:57:56,7,6 ล้อ,89-4045,SB1PL260410023,นครปฐม บางเลน,2026-04-10 09:57:58,N,...,0,NaN,NaN,NaT,NaT,2026-04-10 09:58:32,2026-04-10 09:59:48,NaT,NaT,NaN
3,SB1,Walk in,2026-04-10 09:56:17,6,เทรเลอร์,80-8223,SB1PL260410022,บ.อุดมชัยกู๊ดโฮมสมุทรสงคราม จก.,2026-04-10 09:56:23,N,...,0,NaN,NaN,2026-04-10 10:19:38,2026-04-10 10:20:33,2026-04-10 10:00:25,2026-04-10 10:08:47,NaT,NaT,NaN
4,SB1,Walk in,2026-04-10 09:48:27,5,6 ล้อ,89-4045,SB1PL260410021,บ.จำหน่ายวัตถุก่อสร้าง จก.,2026-04-10 09:48:32,N,...,70,NaN,NaN,NaT,NaT,2026-04-10 09:49:36,NaT,2026-04-10 10:05:27,2026-04-10 10:10:23,NaN


## Save Transformed Data

บันทึกไฟล์ที่แปลงค่าแล้วไว้ใน `data/interim` เพื่อใช้ต่อในขั้น clean

In [112]:
output_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_transformed.csv'
df_transform.to_csv(output_path, index=False, encoding='utf-8-sig')
output_path

WindowsPath('D:/ML/data/interim/vw_timestamp_dashboard_transformed.csv')